# Nuclear shell model

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/shell_model)

Nucleons orbit independently in a Woods–Saxon well. Solving the radial Schrödinger equation for every $(l, j)$ channel as one batch of finite-difference Hamiltonians and turning up the spin–orbit force makes the magic numbers 2, 8, 20, 28, 50, 82, 126 — the nucleon counts of especially stable nuclei — appear as the biggest gaps in the level scheme.

**Vector form**

$$-\frac{\hbar^2}{2m}u'' + \Big[V(r) + \frac{\hbar^2 l(l+1)}{2mr^2}\Big]u = E\,u,\qquad V(r) = -V_0 f(r) + V_{so}\,r_0^2\,\frac1r\frac{df}{dr}\,\langle\mathbf l\cdot\mathbf s\rangle,\quad f = \frac{1}{1+e^{(r-R)/a}}$$

**Index (tensor) form**

$$H_{kij} = -\frac{\hbar^2}{2m}D_{ij} + \Big[V_k(r_i) + \frac{\hbar^2 l_k(l_k+1)}{2m r_i^2}\Big]\delta_{ij},\qquad H_{kij}\,u_{kjn} = E_{kn}\,u_{kin}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $r_i$ | `radial_grid(N, 18.0)` | (N,) | interior grid points; $u = 0$ at $r = 0$ and at the box edge |
| $D_{ij}$ | `second_difference(N, h)` | (N, N) | finite-difference $d^2/dr^2$ |
| $l_k,\ j_k$ | `channels(8)` | (K,) | every channel $(l, j = l \pm \tfrac12)$ up to $l = 8$ |
| $\langle\mathbf l\cdot\mathbf s\rangle_k$ | `ls_factor(ell, j)` | (K,) | $+l/2$ for $j = l+\tfrac12$, $-(l+1)/2$ for $j = l-\tfrac12$ |
| $V_k(r_i)$ | `potential(r, ell, j, ...)` | (K, N) | well + spin–orbit + centrifugal, via einsum('k,i->ki') |
| $H_{kij}$ | `hamiltonian(d2, v)` | (K, N, N) | one Hamiltonian per channel, stacked |
| $E_{kn}$ | `torch.linalg.eigvalsh(H)` | (K, N) | batched over $k$; keep the bound levels $E < 0$ |
| $\sum (2j+1)$ | `cumsum(occupancy(j[k]))` | (M,) | nucleons filled up to each sorted level: the magic numbers sit at the big gaps |

Without spin–orbit the levels bunch like those of a 3D harmonic oscillator and the gaps fall after 2, 8, 20 and 40 nucleons (the flat bottom of the Woods–Saxon well also splits off 58, 92 and 138). The spin–orbit term lowers $j = l + \tfrac12$ and raises $j = l - \tfrac12$ by an amount growing with $l$, so the high-$j$ partner of each oscillator shell (1f7/2, 1g9/2, 1h11/2, 1i13/2) drops into the shell below — Mayer and Jensen's 1949 explanation of 28, 50, 82 and 126. Default parameters are for neutrons in ²⁰⁸Pb: $V_0 = 51 - 33(N-Z)/A \approx 44$ MeV, $r_0 = 1.27$ fm, $a = 0.67$ fm. Simplifications: one fixed well for all levels (no self-consistency, no pairing), no Coulomb term (so these are neutron levels), and a box of 18 fm, so only bound levels are shown.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/shell_model/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    r, h = tensors.radial_grid(params["n_grid"], 18.0, dev)  # (N,) fm
    d2 = tensors.second_difference(len(r), h, dev)  # (N, N)
    ell, j = tensors.channels(8, dev)  # (K,) each
    r0, a = 1.27, params["diffuseness"]
    radius = r0 * params["mass_number"] ** (1 / 3)

    strengths = torch.linspace(0.0, params["v_so"], 40, dtype=torch.float64)  # (S,)
    energies = solve.sweep(r, d2, ell, j, params["v0"], strengths, radius, a, r0)  # (S, K, n_radial)

    for name, e in (("without spin–orbit", energies[0]), (f"V_so = {params['v_so']:.0f} MeV", energies[-1])):
        scheme = solve.level_scheme(e, ell, j)
        _, magic = solve.largest_gaps(scheme, 7)
        print(f"{name}: shell gaps after {sorted(magic.tolist())} nucleons")
    scheme = solve.level_scheme(energies[-1], ell, j)
    print("level      E (MeV)   2j+1   filled")
    for e, k, n, filled in zip(*(scheme[key].tolist() for key in ("energy", "channel", "radial", "filled"))):
        lk, jk = int(ell[k].item()), j[k].item()
        print(f"{tensors.label(n + 1, lk, jk):8s} {e:9.2f}   {int(2 * jk + 1):4d}   {int(filled):6d}")
    return plot.animate(strengths, energies, ell, j, out_dir / "shell_model.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/shell_model/equation.py
r"""Physics → tensor translation.

In the shell model each nucleon moves independently in an average potential: a Woods–Saxon well plus a
spin–orbit term that depends on how the orbital angular momentum l and the spin s are aligned,

    V(r) = -V_0 f(r) + V_{so} \, r_0^2 \, \frac{1}{r}\frac{df}{dr} \, ⟨\mathbf l·\mathbf s⟩,
    f(r) = \frac{1}{1 + e^{(r - R)/a}},    R = r_0 A^{1/3},
    ⟨\mathbf l·\mathbf s⟩ = \tfrac12 [\,j(j+1) - l(l+1) - \tfrac34\,].

Writing ψ = (u(r)/r) × (spin-angle part), each channel k = (l, j) gives a 1D radial equation. On a grid r_i
with the second-difference matrix D_{ij} it is a matrix eigenproblem, one per channel:

    H_{kij} = -\frac{\hbar^2}{2m} D_{ij} + \Big[V_k(r_i) + \frac{\hbar^2 l_k(l_k+1)}{2m r_i^2}\Big] δ_{ij},
    H_{kij} u_{kjn} = E_{kn} u_{kin}        (j summed).

Each level E_{kn} holds 2j + 1 nucleons (the m_j states).
"""

import torch

HBAR2_2M = 20.736  # ħ²/2m for a nucleon in MeV fm²


def woods_saxon_shape(r: torch.Tensor, radius: float, a: float) -> torch.Tensor:
    r"""f(r_i) = 1 / (1 + e^{(r_i − R)/a}).   (N,) → (N,)"""
    return torch.sigmoid((radius - r) / a)


def ls_factor(ell: torch.Tensor, j: torch.Tensor) -> torch.Tensor:
    r"""⟨l·s⟩_k = ½ [j(j+1) − l(l+1) − ¾]: +l/2 for j = l + ½, −(l+1)/2 for j = l − ½.   (K,) → (K,)"""
    return 0.5 * (j * (j + 1) - ell * (ell + 1) - 0.75)


def potential(
    r: torch.Tensor, ell: torch.Tensor, j: torch.Tensor, v0: float, v_so: float, radius: float, a: float, r0: float
) -> torch.Tensor:
    r"""V_k(r_i) + ħ² l_k(l_k+1) / 2m r_i²: well, spin–orbit and centrifugal terms.   → (K, N)"""
    f = woods_saxon_shape(r, radius, a)  # (N,)
    dfdr = -f * (1 - f) / a  # (N,) negative: the well edge
    central = -v0 * f  # (N,)
    spin_orbit = v_so * r0**2 * torch.einsum("k,i->ki", ls_factor(ell, j), dfdr / r)  # (K, N)
    centrifugal = HBAR2_2M * torch.einsum("k,i->ki", ell * (ell + 1), 1 / r**2)  # (K, N)
    return central + spin_orbit + centrifugal


def hamiltonian(d2: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    r"""H_{kij} = −(ħ²/2m) D_{ij} + V_{ki} δ_{ij}.   d2: (N, N), v: (K, N) → (K, N, N)"""
    return -HBAR2_2M * d2 + torch.diag_embed(v)


def occupancy(j: torch.Tensor) -> torch.Tensor:
    """2j + 1 nucleons of one kind fit in a level of angular momentum j."""
    return 2 * j + 1


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/shell_model/tensors.py
"""Build the concrete tensors: radial grid, second-difference matrix and the list of (l, j) channels."""

import torch

L_LETTERS = "spdfghijk"


def radial_grid(n: int, r_max: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """Interior points r_i = h … r_max − h (u = 0 at r = 0 and r = r_max): (N,), spacing h."""
    h = r_max / (n + 1)
    return torch.arange(1, n + 1, dtype=torch.float64, device=device) * h, h


def second_difference(n: int, h: float, device: torch.device) -> torch.Tensor:
    """D_ij u_j ≈ u''(r_i): the tridiagonal (1, −2, 1)/h² matrix, shape (N, N)."""
    main = torch.full((n,), -2.0, dtype=torch.float64, device=device)
    off = torch.ones(n - 1, dtype=torch.float64, device=device)
    return (torch.diag(main) + torch.diag(off, 1) + torch.diag(off, -1)) / h**2


def channels(l_max: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Every (l, j) with j = l ± ½ (only j = ½ for l = 0): two tensors of shape (K,)."""
    ls, js = [], []
    for ell in range(l_max + 1):
        for j in [ell + 0.5, ell - 0.5] if ell > 0 else [0.5]:
            ls.append(float(ell))
            js.append(j)
    as_tensor = {"dtype": torch.float64, "device": device}
    return torch.tensor(ls, **as_tensor), torch.tensor(js, **as_tensor)


def label(n: int, ell: int, j: float) -> str:
    """Spectroscopic name such as 1g9/2 (n counts levels with the same l and j from 1)."""
    return f"{n}{L_LETTERS[ell]}{int(2 * j)}/2"


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/shell_model/solve.py
"""Diagonalise all (l, j) channels at once with a batched eigh, then sort the levels and find the shell gaps."""

import torch

from problems._common.progress import steps

from .equation import hamiltonian, occupancy, potential


def bound_levels(
    r: torch.Tensor,
    d2: torch.Tensor,
    ell: torch.Tensor,
    j: torch.Tensor,
    v0: float,
    v_so: float,
    radius: float,
    a: float,
    r0: float,
    n_radial: int = 6,
) -> torch.Tensor:
    """E_{kn} of the bound states (E < 0) for every channel k: shape (K, n_radial), NaN where unbound."""
    h = hamiltonian(d2, potential(r, ell, j, v0, v_so, radius, a, r0))  # (K, N, N)
    energies = torch.linalg.eigvalsh(h)[:, :n_radial]  # (K, n_radial) lowest levels of each channel
    return torch.where(energies < 0, energies, torch.nan)


def sweep(
    r: torch.Tensor,
    d2: torch.Tensor,
    ell: torch.Tensor,
    j: torch.Tensor,
    v0: float,
    strengths: torch.Tensor,
    radius: float,
    a: float,
    r0: float,
) -> torch.Tensor:
    """Bound levels while the spin–orbit strength is turned up: shape (S, K, n_radial)."""
    return torch.stack(
        [bound_levels(r, d2, ell, j, v0, strengths[s].item(), radius, a, r0) for s in steps(len(strengths), "eigh")]
    )


def level_scheme(energies: torch.Tensor, ell: torch.Tensor, j: torch.Tensor) -> dict[str, torch.Tensor]:
    """Flatten the bound levels, sort them by energy and count nucleons.

    Returns energy, channel index k, radial index n, and the cumulative occupancy Σ (2j + 1) up to and
    including each level: all of shape (M,) for the M bound levels.
    """
    k, n = torch.nonzero(~torch.isnan(energies), as_tuple=True)
    e = energies[k, n]
    order = torch.argsort(e)
    k, n, e = k[order], n[order], e[order]
    return {"energy": e, "channel": k, "radial": n, "filled": torch.cumsum(occupancy(j[k]), dim=0)}


def largest_gaps(scheme: dict[str, torch.Tensor], count: int) -> tuple[torch.Tensor, torch.Tensor]:
    """The `count` biggest jumps between consecutive levels: (gap sizes, nucleon number below each gap)."""
    gaps = torch.diff(scheme["energy"])  # (M − 1,)
    top = torch.argsort(gaps, descending=True)[:count]
    return gaps[top], scheme["filled"][top]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/shell_model/plot.py
"""Left: every level as the spin–orbit strength grows. Right: the level scheme with the shell gaps marked."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .solve import largest_gaps, level_scheme
from .tensors import L_LETTERS, label


def animate(strengths: torch.Tensor, energies: torch.Tensor, ell: torch.Tensor, j: torch.Tensor, path: Path) -> Path:
    """strengths (S,), energies (S, K, n_radial) with NaN for unbound levels."""
    s = strengths.cpu().numpy()
    e = energies.cpu().numpy()
    l_np, j_np = ell.cpu().numpy().astype(int), j.cpu().numpy()
    e_min = np.nanmin(e) * 1.15  # room for the legend below the deepest level

    fig, (ax_s, ax_l) = plt.subplots(1, 2, figsize=(8.5, 5.5), gridspec_kw={"width_ratios": [1.3, 1]}, sharey=True)
    ax_s.set_xlim(0, max(s[-1], 1e-9))
    ax_s.set_ylim(e_min, 0)
    ax_s.set_xlabel(r"spin–orbit strength $V_{so}$ (MeV)")
    ax_s.set_ylabel("single-particle energy E (MeV)")
    traces = {}
    for k in range(e.shape[1]):
        for n in range(e.shape[2]):
            if not np.all(np.isnan(e[:, k, n])):
                style = "-" if j_np[k] > l_np[k] else "--"  # solid j = l + ½, dashed j = l − ½
                traces[(k, n)] = ax_s.plot([], [], style, color=ACCENT[l_np[k] % len(ACCENT)], lw=1.2)[0]
    for ll in range(int(l_np.max()) + 1):
        ax_s.plot([], [], color=ACCENT[ll % len(ACCENT)], lw=2, label=L_LETTERS[ll])
    ax_s.legend(loc="lower left", fontsize=7, ncol=5, title="l (solid: j = l+½)", title_fontsize=7)

    ax_l.set_xlim(0, 1)
    ax_l.set_xticks([])
    ax_l.grid(False)
    drawn: list = []

    def update(step: int):
        for (k, n), line in traces.items():
            line.set_data(s[: step + 1], e[: step + 1, k, n])
        for artist in drawn:
            artist.remove()
        drawn.clear()
        scheme = level_scheme(energies[step], ell, j)
        energy = scheme["energy"].cpu().numpy()
        label_y = _spread(energy, 0.022 * abs(e_min), top=-0.01 * abs(e_min))
        for i, (en, k, n) in enumerate(zip(energy, scheme["channel"].tolist(), scheme["radial"].tolist())):
            colour = ACCENT[l_np[k] % len(ACCENT)]
            drawn.extend(ax_l.plot([0.05, 0.4], [en, en], color=colour, lw=2))
            drawn.extend(ax_l.plot([0.4, 0.47], [en, label_y[i]], color="#d1d5db", lw=0.6))
            drawn.append(ax_l.text(0.48, label_y[i], label(n + 1, l_np[k], j_np[k]), va="center", fontsize=6))
        gaps, filled = largest_gaps(scheme, 7)
        for number in filled.tolist():
            i = int(np.searchsorted(scheme["filled"].cpu().numpy(), number))
            mid = 0.5 * (energy[i] + energy[i + 1])
            drawn.append(
                ax_l.text(0.82, mid, f"{int(number)}", color="#b45309", fontsize=10, weight="bold", va="center")
            )
        ax_l.set_title(f"$V_{{so}}$ = {s[step]:.1f} MeV  (gold: 7 largest gaps)", fontsize=9)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(s), 60), path, fps=10)


def _spread(y: np.ndarray, gap: float, top: float = 0.0) -> np.ndarray:
    """Push sorted label positions apart so neighbouring labels never overlap, keeping them below `top`."""
    out = y.copy()
    for i in range(1, len(out)):  # upwards: each label at least `gap` above the previous one
        out[i] = max(out[i], out[i - 1] + gap)
    out[-1] = min(out[-1], top)
    for i in range(len(out) - 2, -1, -1):  # downwards: squeeze back under the top
        out[i] = min(out[i], out[i + 1] - gap)
    return out


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'v_so': 22.0,
    'v0': 44.0,
    'mass_number': 208,
    'diffuseness': 0.67,
    'n_grid': 300
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.shell_model")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.shell_model.main")
problem_dir = Path("problems/nuclear/shell_model")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
